# INSID3 — Training-Free In-Context Segmentation (Tutorial 01: Natural One-Shot)

**INSID3** is a training-free, one-shot segmentation algorithm from CVPR 2026 Oral.
It uses a frozen DINOv3 backbone to transfer a segmentation mask from a
reference image to a query image — no fine-tuning required.

**Paper**: arXiv 2603.28480 | **Code**: visinf/INSID3 (Apache-2.0)
**Backbone**: DINOv3 (Meta custom license — BYOT, user accepts upstream)

**Attribution**: "Built with DINOv3" is required by the DINOv3 license.

**Prerequisites**:
1. Accept the DINOv3 upstream license on Hugging Face
2. Run `huggingface-cli login` to cache your token
3. `pip install 'visionservex[hf]' Pillow scikit-learn`

In [ ]:
# Check INSID3 policy and HF access before running
from visionservex.vsx import VSX

handle = VSX.insid3('insid3-large')
info = handle.explain()
print('Model:', info['model_id'])
print('Task:', info['task'])
print('Policy:', info['state'])
print('Attribution required:', info['attribution_required'])
print('HF repo:', info['hf_repo'])

In [ ]:
# Check doctor (HF token + backbone accessibility)
import subprocess
result = subprocess.run(
    ['visionservex', 'insid3', 'doctor', '--model-id', 'insid3-large'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('BLOCKED:', result.stderr)
    print('Run: huggingface-cli login')

In [ ]:
# Create synthetic demo images (replace with real images for real results)
from PIL import Image, ImageDraw
import tempfile, os

# Reference: 256x256 image with an orange circle
ref_img = Image.new('RGB', (256, 256), color=(50, 50, 100))
draw = ImageDraw.Draw(ref_img)
draw.ellipse([80, 80, 176, 176], fill=(220, 120, 40))

# Reference mask: white circle where the object is
ref_mask = Image.new('L', (256, 256), color=0)
draw_m = ImageDraw.Draw(ref_mask)
draw_m.ellipse([80, 80, 176, 176], fill=255)

# Query: 256x256 with a similar circle at a different position
qry_img = Image.new('RGB', (256, 256), color=(70, 40, 80))
draw_q = ImageDraw.Draw(qry_img)
draw_q.ellipse([140, 60, 220, 140], fill=(210, 110, 50))

tmp_dir = tempfile.mkdtemp()
ref_path = os.path.join(tmp_dir, 'ref.jpg')
ref_mask_path = os.path.join(tmp_dir, 'ref_mask.png')
qry_path = os.path.join(tmp_dir, 'query.jpg')

ref_img.save(ref_path)
ref_mask.save(ref_mask_path)
qry_img.save(qry_path)
print(f'Demo images saved to: {tmp_dir}')

In [ ]:
# Run INSID3 segmentation
out_dir = os.path.join(tmp_dir, 'insid3_out')

result = handle.segment(
    qry_path, ref_path, ref_mask_path,
    device='cpu',
    n_clusters=6,
    out_dir=out_dir,
)

print('Status:', result['status'])
print('State:', result['state'])
if result['status'] == 'ok':
    print('Mask area (px):', result['mask_area_px'])
    print('Best cluster similarity:', result['best_cluster_sim'])
    print('Load time (ms):', result['load_ms'])
    print('Inference time (ms):', result['feat_ms'] + result['seg_ms'])
    print('Attribution required:', result['attribution_required'])
elif result['status'] == 'blocked':
    print('Blocked reason:', result['reason'])
    print('Next command:', result.get('next_command', ''))

In [ ]:
# Display results (if segmentation ran)
import os
if result.get('status') == 'ok' and result.get('saved_paths'):
    from IPython.display import display
    pred_mask = Image.open(result['saved_paths']['pred_mask'])
    overlay = Image.open(result['saved_paths']['overlay'])
    print('Query image | Reference | Predicted mask | Overlay')
    display(qry_img, ref_img, pred_mask, overlay)
else:
    print('No output saved — check status above')

## What just happened?

1. DINOv3-ViT-L/16 extracted patch-level features from both images
2. Positional debiasing (SVD) removed position-dependent components
3. Reference masked patches were averaged into a prototype vector
4. Query patches were clustered (agglomerative); the cluster nearest the prototype was selected
5. The winning cluster mask was upsampled to the original image resolution

**No training, no fine-tuning, no INSID3-specific weights.**

**License reminder**: This output was produced with DINOv3 backbone weights.
If you use this in a product, include "Built with DINOv3" attribution per the DINOv3 License.